# Hotspot Analysis with Getis-Ord G and G*

This notebook covers Getis-Ord statistics for identifying spatial concentrations of high or low values.

## Learning goals

By the end of this notebook, you will be able to:

- explain how Getis-Ord $G$ differs from Moran's $I$
- compute the global $G$ statistic and assess its significance
- compute local $G_i^*$ statistics to identify individual hotspots and coldspots
- map statistically significant hotspots and coldspots

The substantive question is whether high Airbnb prices cluster spatially — and, if so, *where* those high-price hotspots are located.

In [ ]:
import geopandas as gpd
import libpysal as lps
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import esda

## Data preparation

We use the same Berlin neighborhood dataset as in the global Moran's I and local Moran's I notebooks so that results can be compared directly.

In [ ]:
gdf = gpd.read_file("data/berlin-neighbourhoods.geojson")

In [ ]:
bl_df = pd.read_csv("data/berlin-listings.csv")
geometry = gpd.points_from_xy(x=bl_df.longitude, y=bl_df.latitude, crs="epsg:4326")
bl_gdf = gpd.GeoDataFrame(bl_df, geometry=geometry)

In [ ]:
bl_gdf["price"] = bl_gdf["price"].astype("float32")
sj_gdf = gpd.sjoin(
    gdf, bl_gdf, how="inner", predicate="intersects", lsuffix="left", rsuffix="right"
)
median_price_gb = sj_gdf["price"].groupby([sj_gdf["neighbourhood_group"]]).mean()
gdf = gdf.join(median_price_gb, on="neighbourhood_group")
gdf.rename(columns={"price": "median_pri"}, inplace=True)
gdf["median_pri"] = gdf["median_pri"].fillna(gdf["median_pri"].mean())

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10), subplot_kw={"aspect": "equal"})
gdf.plot(column="median_pri", scheme="Quantiles", k=5, cmap="OrRd", legend=True, ax=ax)
ax.set_axis_off()
ax.set_title("Median Airbnb price by neighbourhood (quintiles)")
plt.show()

## Why Getis-Ord G?

Moran's $I$ measures spatial autocorrelation by comparing each observation to its *deviation from the global mean*. A High-High cluster in Moran's framework means both a neighborhood and its neighbors are above the mean.

Getis-Ord $G$ takes a different approach: it measures spatial concentration of *raw values*. A high $G$ means large values are co-located with other large values — a true **hotspot**. Because $G$ works with raw values rather than mean-centered deviations, it is particularly well-suited to detecting concentration of phenomena that are always non-negative (counts, prices, rates).

The global statistic $G$ answers: *is there a statistically significant overall concentration of high or low values?* The local statistic $G_i^*$ answers: *which specific locations are part of a hotspot or coldspot?*

## Global G

The global statistic tests whether the entire study area exhibits significant spatial concentration. A high $G$ relative to its expectation suggests high values cluster together; a low $G$ suggests high and low values are interleaved.

In [ ]:
df = gdf
# G requires binary (not row-standardised) weights
wq = lps.weights.Queen.from_dataframe(df, use_index=False, silence_warnings=True)
wq.transform = "b"
y = df["median_pri"]

In [ ]:
np.random.seed(12345)
g_global = esda.G(y, wq)
print(f"Observed G:   {g_global.G:.6f}")
print(f"Expected G:   {g_global.EG:.6f}")
print(f"z-score:      {g_global.z_norm:.4f}")
print(f"p-value (norm): {g_global.p_norm:.4f}")
print(f"p-value (sim):  {g_global.p_sim:.4f}")

## Permutation-based inference

As with Moran's $I$, we assess the observed $G$ against a reference distribution generated by randomly permuting attribute values while holding the spatial structure fixed.

In [ ]:
import seaborn as sns

sns.kdeplot(g_global.sim, fill=True)
plt.vlines(g_global.G, 0, plt.gca().get_ylim()[1] * 0.9, color="r", label="Observed G")
plt.vlines(g_global.EG, 0, plt.gca().get_ylim()[1] * 0.9, color="k", linestyle="--", label="Expected G")
plt.xlabel("G statistic")
plt.legend()
plt.title("Permutation reference distribution for global G")
plt.show()

## From global to local: $G_i^*$

The global $G$ tells us *whether* concentration exists; $G_i^*$ (G-star) tells us *where*. For each location $i$, $G_i^*$ computes the ratio of the sum of values in $i$'s neighbourhood (including $i$ itself) to the sum of all values. A large positive $z$-score flags a **hotspot** — a neighbourhood surrounded by other high-value neighbourhoods. A large negative $z$-score flags a **coldspot**.

The key difference from Moran's I quadrant labels is that $G_i^*$ only distinguishes hotspots from coldspots; it does not identify spatial outliers (High-Low or Low-High).

## Computing $G_i^*$

We pass `star=True` to include the focal observation in the local sum, producing $G_i^*$ (the more commonly reported variant). Weights must be binary.

In [ ]:
np.random.seed(12345)
g_local = esda.G_Local(y, wq, star=True, seed=12345)
print(f"Significant hotspots (p<0.05): {(g_local.p_sim < 0.05).sum()}")

### Mapping $G_i^*$ z-scores

The z-score map gives a continuous view of local concentration. Red (large positive) indicates hotspot cores; blue (large negative) indicates coldspot cores.

In [ ]:
df = df.copy()
df["Zs"] = g_local.Zs

fig, ax = plt.subplots(figsize=(12, 10), subplot_kw={"aspect": "equal"})
df.plot(
    column="Zs",
    cmap="RdBu_r",
    legend=True,
    ax=ax,
    vmin=-3,
    vmax=3,
)
ax.set_axis_off()
ax.set_title("Local $G_i^*$ z-scores")
plt.show()

### Mapping significant hotspots and coldspots

By applying a significance threshold (p < 0.05) we retain only the statistically credible clusters.

In [ ]:
sig = g_local.p_sim < 0.05
hotspot = sig & (g_local.Zs > 0)
coldspot = sig & (g_local.Zs < 0)

spot_labels = pd.Series("not significant", index=df.index)
spot_labels[hotspot] = "hotspot"
spot_labels[coldspot] = "coldspot"
df["spot"] = spot_labels

colors = {"hotspot": "red", "coldspot": "blue", "not significant": "lightgrey"}
df["color"] = df["spot"].map(colors)

fig, ax = plt.subplots(figsize=(12, 10), subplot_kw={"aspect": "equal"})
for label, color in colors.items():
    df[df["spot"] == label].plot(ax=ax, color=color, label=label)
ax.legend()
ax.set_axis_off()
ax.set_title("Significant $G_i^*$ hotspots and coldspots (p < 0.05)")
plt.show()

### Comparing Getis-Ord and Moran's I results

Hotspots identified by $G_i^*$ correspond approximately to the High-High cluster quadrant from local Moran's $I$, but the two methods can diverge when local variance is high — a neighborhood with very high prices surrounded by moderate prices may register as a spatial outlier in Moran but not a hotspot in $G_i^*$.

## Takeaways

- **Global G** tests whether the entire study area has more concentration of high (or low) values than expected under spatial randomness.
- **Local $G_i^*$** assigns each location a z-score that reflects whether it sits in a hotspot or coldspot.
- Getis-Ord works on *raw values*, making it appropriate for non-negative quantities (prices, counts, rates).
- Unlike local Moran's $I$, $G_i^*$ does not identify spatial outliers — only high-value and low-value clusters.
- Permutation inference avoids distributional assumptions and is the recommended approach for both the global and local statistics.